# Log-sum-exp pooling: 29 layers to 15 windows, PCA to 512 per layer, then 4 × 128 MLP

Identical to the log-sum-exp 4 × 128 run (same pooling, six-source split, BCE loss, 40 epochs, checkpoint selection, held-out TriviaQA evaluation), except that **each of the 15 pooled layers gets its own PCA (fitted on source-train rows only) from 3,584 to 512 dimensions**; the 15 × 512 = 7,680 components are concatenated and fed to the MLP.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout c42fa95
!pip install -q -r requirements-kaggle.txt

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1: raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Saved features:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
OUTPUT_DIR = Path('/kaggle/working/sala_g14_6src_logsumexp_layerpca512_4x128')
command = [sys.executable, '/kaggle/working/SALA/sala_raw_concat.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(OUTPUT_DIR), '--bleurt-threshold', '0.5', '--epochs-erm', '40', '--seed', '42', '--train-device', 'cuda:0', '--hidden-dim', '128', '--hidden-layers', '4', '--pooling', 'logsumexp1d', '--pool-kernel-size', '3', '--pool-stride', '2', '--feature-pca-dim', '512', '--feature-pca-per-layer']
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
results = pd.read_csv(OUTPUT_DIR / 'results.csv')
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
print('PCA dim:', summary['feature_pca_dim'], 'explained variance:', summary['feature_pca_explained_variance_total'])
print('Pooled layers:', summary['pooled_layers'], 'channels:', summary['pooling_output_channels'], 'MLP input:', summary['mlp_input_dimension'])
print(json.dumps(summary['target_metrics'], indent=2))
display(results.head())